In [2]:
import pandas as pd
import numpy as np
import altair as alt
from vega_datasets import data
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [3]:
#race = pd.read_csv('data/od_race.csv')
#inc = pd.read_csv('data/od_inc.csv')
race_inc = pd.read_csv('../data/od.csv')
agg = pd.read_csv('../data/od_pooled.csv')

#print(race.head())

In [4]:
home = agg[agg["o_cz"] == agg["d_cz"]].copy()

home.head()

,o_cz,o_cz_name,o_state_name,d_cz,d_cz_name,d_state_name,n,n_tot_o,n_tot_d,pool,pr_d_o,pr_o_d
170,100,Johnson City,Tennessee,100,Johnson City,Tennessee,42829.0,57198,56915,pooled,0.748785,0.752508
1362,200,Morristown,Tennessee,200,Morristown,Tennessee,16023.0,24922,24188,pooled,0.642926,0.662436
2166,301,Middlesborough,Tennessee,301,Middlesborough,Tennessee,4207.0,6539,6035,pooled,0.643371,0.697100
2484,302,Knoxville,Tennessee,302,Knoxville,Tennessee,55022.0,76167,84432,pooled,0.722386,0.651672
3232,401,Winston-Salem,North Carolina,401,Winston-Salem,North Carolina,33790.0,53108,52381,pooled,0.636251,0.645081


In [5]:
home["share_born_here"] = home["pr_o_d"]
home["share_not_from_here"] = 1 - home["pr_o_d"]
home["share_left"] = 1 - home["pr_d_o"]

large = home.nlargest(50, "n_tot_o")

In [21]:
chart = alt.Chart(large).mark_bar().encode(
    x=alt.X("share_not_from_here:Q", axis=alt.Axis(format="%")),
    y=alt.Y("o_cz_name:N", sort="-x"),
)


In [7]:
chart = alt.Chart(large).mark_bar().encode(
    x=alt.X("share_left:Q", axis=alt.Axis(format="%")),
    y=alt.Y("o_cz_name:N", sort="-x"),
)

In [8]:
not_from = (alt.Chart(large).transform_calculate(avg = "(datum.share_left + datum.share_not_from_here) /2")
.mark_bar(color = 'gold').encode(
    x=alt.X("share_not_from_here:Q", axis=alt.Axis(format="%"), 
    sort = 'descending', title = "Current Residents Not From Here",
    scale = alt.Scale(domain = [0,0.6])),
    y=alt.Y("o_cz_name:N", sort = alt.EncodingSortField(field = "avg", order = 'descending'), title = "Metro Area")
))

left = (alt.Chart(large).transform_calculate(avg = "(datum.share_left + datum.share_not_from_here) /2")
.mark_bar(color = 'green').encode(
    x=alt.X("share_left:Q", axis=alt.Axis(format="%"), title = "Original Residents Who Left",
    scale = alt.Scale(domain = [0, 0.6])),
    y=alt.Y("o_cz_name:N", axis = None, sort = alt.EncodingSortField(field = "avg", order = 'descending')),
))

chart = alt.concat(not_from, left, spacing = 0)

chart.save('graphs/graph1.png')

In [9]:
large[large['d_cz_name'] == 'Port St. Lucie']

,o_cz,o_cz_name,o_state_name,d_cz,d_cz_name,d_state_name,n,n_tot_o,n_tot_d,pool,pr_d_o,pr_o_d,share_born_here,share_not_from_here,share_left
76000,7100,Port St. Lucie,Florida,7100,Port St. Lucie,Florida,91052.0,147427,140798,pooled,0.617607,0.646685,0.646685,0.353315,0.382393


In [10]:
home["gap"] = home["share_not_from_here"] - home["share_left"]

In [22]:
graph = alt.Chart(home).mark_circle(color = 'purple').encode(
    x=alt.X("n_tot_o:Q", scale=alt.Scale(type="log"), title = "Total Residents of Origin"),
    y=alt.Y("share_left:Q", axis=alt.Axis(format="%"), title = "Original Residents Who Left"))

#graph.save('graphs/graph2.png')
graph

alt.Chart(...)

In [23]:
points = alt.Chart(home[home["n_tot_d"] > 0]).mark_circle(color = 'purple').encode(
    x=alt.X("share_left:Q", axis=alt.Axis(format="%"), title = "Current Residents Not From Here"),
    y=alt.Y("share_not_from_here:Q", axis=alt.Axis(format="%"), title = "Original Residents Who Left"),
    size="n_tot_d:Q",
    tooltip=["o_cz_name", "o_state_name", "gap"],
)

line_data = pd.DataFrame({
    "share_left": [0,1],
    "share_not_from_here": [0,1]
})

line = alt.Chart(line_data).mark_line(color = 'gray').encode(
    x = "share_left:Q",
    y = "share_not_from_here:Q"
)

graph = points + line

#graph.save('graphs/graph3.png')
graph

alt.LayerChart(...)

In [ ]:
#alt.themes.enable('dark')

chi_out = race[(race["o_cz_name"] == "Chicago") & (race["d_cz_name"] != "Chicago")]
top = (chi_out.sort_values("pr_d_o", ascending=False)
          .groupby("pool").head(8)["d_cz_name"].unique())

graph = alt.Chart(chi_out[chi_out["d_cz_name"].isin(top)]).mark_circle(size=90).encode(
    x=alt.X("pr_d_o:Q", axis=alt.Axis(format="%"), title = "Probability of living in destination given they are from Chicago"),
    y=alt.Y("d_cz_name:N", sort="-x", title = "Destination Metro Area"),
    color=alt.Color("pool:N").title("Race"),
)

#graph.save('graphs/graph4.png')
graph

alt.Chart(...)

In [ ]:
#alt.themes.enable('dark')

chi_in = race[(race["d_cz_name"] == "Chicago") & (race["o_cz_name"] != "Chicago")]
top = (chi_in.sort_values("pr_o_d", ascending=False)
          .groupby("pool").head(8)["o_cz_name"].unique())

graph = alt.Chart(chi_in[chi_in["o_cz_name"].isin(top)]).mark_circle(size=90).encode(
    x=alt.X("pr_o_d:Q", axis=alt.Axis(format="%"), title = "Probability of being from origin given they live in Chicago"),
    y=alt.Y("o_cz_name:N", sort="-x", title = "Origin Metro Area"),
    color=alt.Color("pool:N").legend(title = "Race"),
)

#graph.save('graphs/graph5.png')
graph

alt.Chart(...)

In [ ]:
#alt.themes.enable('fivethirtyeight')

chi_out = inc[(inc["o_cz_name"] == "Chicago") & (inc["d_cz_name"] != "Chicago")]
top = (chi_out.sort_values("pr_d_o", ascending=False)
          .groupby("pool").head(8)["d_cz_name"].unique())

graph = alt.Chart(chi_out[chi_out["d_cz_name"].isin(top)]).mark_tick().encode(
    x=alt.X("pr_d_o:Q", axis=alt.Axis(format="%"), title = "Probability of living in destination given they are from Chicago"),
    y=alt.Y("d_cz_name:N", sort="-x", title = 'Destination Metro Area'),
    color=alt.Color("pool:N").legend(title = "Income Quartile"),
).configure_tick(
    thickness = 2
)

#graph.save('graphs/graph6.png')
graph

alt.Chart(...)

In [27]:
chi_in = inc[(inc["d_cz_name"] == "Chicago") & (inc["o_cz_name"] != "Chicago")]
top = (chi_in.sort_values("pr_o_d", ascending=False)
          .groupby("pool").head(8)["o_cz_name"].unique())

graph = alt.Chart(chi_in[chi_in["o_cz_name"].isin(top)]).mark_tick().encode(
    x=alt.X("pr_o_d:Q", axis=alt.Axis(format="%")),
    y=alt.Y("o_cz_name:N", sort="-x"),
    color="pool:N",
).configure_tick(
    thickness = 3
)

#graph.save('graphs/graph7.png')
graph

alt.Chart(...)

In [4]:
race_inc[["race", "income_quintile"]] = race_inc["pool"].str.extract(r"^(.*?)Q(\d)$")

race_inc["stayed"] = race_inc["n"].where(race_inc["o_cz"] == race_inc["d_cz"], 0)

inc_race_matrix = (race_inc.groupby(["race", "income_quintile"], as_index=False)
           .agg(total=("n", "sum"), stayed=("stayed", "sum")))
inc_race_matrix["share_left"] = 1 - inc_race_matrix["stayed"] / inc_race_matrix["total"]

inc_race_matrix

,race,income_quintile,total,stayed,share_left
0,Asian,1,86064,63599,0.261027
1,Asian,2,126807,95921,0.243567
2,Asian,3,158092,118443,0.250797
3,Asian,4,159519,119633,0.250039
4,Asian,5,214622,150168,0.300314
5,Black,1,1864054,1404324,0.246629
6,Black,2,1443612,1085483,0.248078
7,Black,3,589861,425555,0.278550
8,Black,4,354029,253287,0.284559
9,Black,5,224258,152868,0.318339


In [6]:
race_inc

,o_cz,o_cz_name,o_state_name,d_cz,d_cz_name,d_state_name,n,n_tot_o,n_tot_d,pool,pr_d_o,pr_o_d,race,income_quintile,stayed
0,100,Johnson City,Tennessee,10000,Panama City,Florida,0,36,44,AsianQ1,0.000000,0.000000,Asian,1,0
1,100,Johnson City,Tennessee,31303,Kerrville,Texas,0,36,2,AsianQ1,0.000000,0.000000,Asian,1,0
2,100,Johnson City,Tennessee,10600,Jasper,Alabama,0,36,6,AsianQ1,0.000000,0.000000,Asian,1,0
3,100,Johnson City,Tennessee,2600,Roanoke Rapids,North Carolina,0,36,8,AsianQ1,0.000000,0.000000,Asian,1,0
4,100,Johnson City,Tennessee,11402,Marinette,Michigan,0,36,5,AsianQ1,0.000000,0.000000,Asian,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13727020,39400,Seattle,Washington,22800,Eau Claire,Wisconsin,1,96029,4638,WhiteQ5,0.000010,0.000216,White,5,0
13727021,39400,Seattle,Washington,16500,Erie,Pennsylvania,7,96029,5811,WhiteQ5,0.000073,0.001205,White,5,0
13727022,39400,Seattle,Washington,33601,Lawton,Oklahoma,11,96029,1806,WhiteQ5,0.000115,0.006091,White,5,0
13727023,39400,Seattle,Washington,38601,Spokane,Washington,924,96029,11179,WhiteQ5,0.009622,0.082655,White,5,0


In [5]:
graph = alt.Chart(inc_race_matrix).mark_rect().encode(
    alt.X("race:N", title = 'Race'),
    alt.Y("income_quintile:O", title = 'Income Quintile'),
    alt.Color('share_left', title = 'Share of Original Residents Who Left')
)

#graph.save('graphs/graph8.png')
graph

alt.Chart(...)

In [29]:
statewide = (agg.groupby(['o_state_name', 'd_state_name'], as_index = False)["n"].sum())

graph = alt.Chart(statewide).mark_rect().encode(
    alt.X("o_state_name:N", title = 'Origin State'),
    alt.Y("d_state_name:N", title = "Destination State"),
    alt.Color('n:Q').legend(title = 'Number of Residents')
).transform_filter(
    'datum.o_state_name != datum.d_state_name'
)

#graph.save('graphs/graph9.png')
graph

alt.Chart(...)

In [30]:
cz = pd.read_excel('data/czlma903.xls')

chi = agg[agg["d_cz_name"] == "Chicago"]
chicago_id = chi.loc[chi["o_cz_name"] == "Chicago", "o_cz"].iloc[0]


cz["fips"] = cz["County FIPS Code"].astype(int)
cz = cz.rename(columns={"CZ90": "cz"})[["fips", "cz"]]

cz_vals = cz.merge(chi, left_on="cz", right_on="o_cz", how="left")
cz_vals["pr_o_d"] = cz_vals["pr_o_d"].fillna(0)

counties = alt.topo_feature(data.us_10m.url, "counties")

color_max = cz_vals.loc[cz_vals["o_cz"] != chicago_id, "pr_o_d"].max()

graph = (alt.Chart(counties)
   .mark_geoshape(stroke="#d9d9d9", strokeWidth=0.2)
   .transform_lookup(
       lookup="id",
       from_=alt.LookupData(cz_vals, "fips", ["pr_o_d", "o_cz", "o_cz_name"]))
   .encode(
       color=alt.Color("pr_o_d:Q",
                       scale=alt.Scale(type="symlog", constant=color_max / 200,
                                       domain=[0, color_max], clamp=True,
                                       range=["#ffffff", "#08306b"]),
                       legend=alt.Legend(title=["Share of Chicago residents","from home county"], format=".2%")))
   .project("albersUsa")
   .properties(width=700, height=420))

#graph.save('graphs/graph10.png')
graph

alt.Chart(...)

1. Maybe do an overview of population in the US - join in population data 
2. maybe emphasize differences between rural and urban areas
3. label as location at 16 vs 26?
4. calculate percentages out of people who aren't/are from there
5. each origin state's highest destination state - in heatmap?
6. flows (sankey?) of people moving between regions and urbanicity

In [40]:
race_inc[["race", "income_quintile"]] = race_inc["pool"].str.extract(r"^(.*?)Q(\d)$")

keys = ["o_cz", "o_cz_name", "o_state_name", "d_cz", "d_cz_name", "d_state_name"]

# Counts by race
race_w = race_inc.pivot_table(index=keys, columns="race", values="n",
                        aggfunc="sum", fill_value=0)
race_w.columns = [f"n_{c.lower()}" for c in race_w.columns]

# Counts by income quintile
inc_w = race_inc.pivot_table(index=keys, columns="income_quintile", values="n",
                       aggfunc="sum", fill_value=0)
inc_w.columns = [f"n_q{c}" for c in inc_w.columns]

wide = race_w.join(inc_w).reset_index()

In [41]:
race_cols = list(race_w.columns)
inc_cols = list(inc_w.columns)
wide["n_total"] = wide[race_cols].sum(axis=1)

In [42]:
#n = number of individuals from origin living in destination 
wide["n_tot_o"] = wide.groupby("o_cz")["n_total"].transform("sum")   # everyone who grew up in o
wide["n_tot_d"] = wide.groupby("d_cz")["n_total"].transform("sum")   #everyone who lives in d

wide

,o_cz,o_cz_name,o_state_name,d_cz,d_cz_name,d_state_name,n_asian,n_black,n_hispanic,n_other,n_white,n_q1,n_q2,n_q3,n_q4,n_q5,n_total,n_tot_o,n_tot_d
0,100,Johnson City,Tennessee,100,Johnson City,Tennessee,99,883,504,1227,40116,9046,9199,12173,8838,3573,42829,57198,56915
1,100,Johnson City,Tennessee,200,Morristown,Tennessee,0,17,6,27,672,188,165,193,133,43,722,57198,24188
2,100,Johnson City,Tennessee,301,Middlesborough,Tennessee,0,0,0,1,59,15,13,20,7,5,60,57198,6035
3,100,Johnson City,Tennessee,302,Knoxville,Tennessee,3,33,16,48,1372,161,225,309,411,366,1472,57198,84432
4,100,Johnson City,Tennessee,401,Winston-Salem,North Carolina,0,4,1,2,92,24,14,19,20,22,99,57198,52381
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
549076,39400,Seattle,Washington,39205,John Day,Oregon,0,0,0,1,8,3,2,2,2,0,9,402101,591
549077,39400,Seattle,Washington,39301,Friday Harbor,Washington,2,1,6,18,171,25,27,47,56,43,198,402101,1063
549078,39400,Seattle,Washington,39302,Bellingham,Washington,86,55,246,347,3629,400,625,844,1133,1361,4363,402101,21813
549079,39400,Seattle,Washington,39303,Port Angeles,Washington,7,8,56,107,818,148,179,244,244,181,996,402101,8009


In [ ]:
statewide['n_tot_o'] = statewide.groupby('o_state_name')['n'].transform('sum')
statewide['n_tot_d'] = statewide.groupby('d_state_name')['n'].transform('sum')
statewide['']

statewide

,o_state_name,d_state_name,n,n_tot_o,n_tot_d
0,Alabama,Alabama,399099.0,496176.0,483938.0
1,Alabama,Alaska,418.0,496176.0,92643.0
2,Alabama,Arizona,949.0,496176.0,610247.0
3,Alabama,Arkansas,913.0,496176.0,317759.0
4,Alabama,California,4154.0,496176.0,3554199.0
...,...,...,...,...,...
2596,Wyoming,Virginia,189.0,67369.0,607630.0
2597,Wyoming,Washington,1138.0,67369.0,707186.0
2598,Wyoming,West Virginia,33.0,67369.0,132110.0
2599,Wyoming,Wisconsin,186.0,67369.0,582815.0
